# RAG with LlamaIndex: Indexing, Retrieval, Optimisation & LLM Connection

LlamaIndex is a high-level data framework for building RAG applications. It abstracts away the differences between vector stores, embedding models, and LLMs behind a unified API, letting you focus on the pipeline logic rather than boilerplate.

This notebook covers:
- Loading and chunking a PDF via LlamaIndex's built-in readers and parsers
- Building indexes over FAISS, ChromaDB, and LlamaIndex's native `StorageContext` persistence
- Retrieval strategies (similarity, MMR, hybrid, reranking with `QueryFusionRetriever`)
- Index optimisation (chunk size, embedding model, top-k, reranking)
- Connecting an LLM with RAG to reduce context token usage
- Comparing multiple vector stores from a single API

In [ ]:
from __future__ import annotations

import os
import shutil
import time
from pathlib import Path

import numpy as np

# LlamaIndex core
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    StorageContext,
    load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import (
    SimilarityPostprocessor,
    SentenceTransformerRerank,
)

# Embedding models
from llama_index.embeddings.ollama import OllamaEmbedding

# LLM
from llama_index.llms.ollama import Ollama

# Vector stores
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.vector_stores.faiss import FaissVectorStore

import chromadb
import faiss

# Optional metadata filtering
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator

print("All imports OK")

## 1. Document Loading and Chunking

LlamaIndex's `SimpleDirectoryReader` loads all supported files (PDF, DOCX, images, etc.) from a directory. We then split into `Node` objects (the LlamaIndex equivalent of chunks).

In [ ]:
def find_rag_directory() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        direct = candidate / 'doc' / 'NVDA_report.pdf'
        nested = candidate / 'ai_tutorials' / 'rag' / 'doc' / 'NVDA_report.pdf'
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / 'ai_tutorials' / 'rag'
    raise FileNotFoundError('Could not find NVDA_report.pdf')

RAG_DIR = find_rag_directory()
DOC_PATH = RAG_DIR / 'doc'

# Use SimpleDirectoryReader
documents = SimpleDirectoryReader(
    input_dir=str(DOC_PATH),
    required_exts=[".pdf"],
).load_data()
print(f"Loaded {len(documents)} document(s)")

# Chunk into nodes
parser = SentenceSplitter(chunk_size=900, chunk_overlap=150)
nodes = parser.get_nodes_from_documents(documents)
print(f"Created {len(nodes)} nodes")
print(f"\nFirst node preview:\n{nodes[0].text[:200]}...")

## 2. Global Settings (Embedding + LLM)

LlamaIndex uses a global `Settings` object. Set the embedding model and LLM once, and everything downstream picks them up automatically.

In [ ]:
# Embedding model
Settings.embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)

# LLM
Settings.llm = Ollama(
    model="llama3.1",
    base_url="http://localhost:11434",
    temperature=0,
)

# Verify
test_vec = Settings.embed_model.get_text_embedding("test")
print(f"Embedding dimension: {len(test_vec)}")
print(f"Embed model: {Settings.embed_model.model_name}")
print(f"LLM model:   {Settings.llm.model}")

## 3. Building Indexes Over Different Vector Stores

LlamaIndex provides a uniform `VectorStoreIndex` class. You plug in different vector stores via `StorageContext` without changing any other code.

We build three indexes:
- **A**: ChromaDB (persistent, zero-config)
- **B**: FAISS (high-performance)
- **C**: Native LlamaIndex JSON persistence (no external vector store)

All three use the same embedding model and same nodes.

In [ ]:
# === Index A: ChromaDB ===
CHROMA_DIR = RAG_DIR / 'vector_db_llamaindex' / 'chroma'
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("Building ChromaDB index...")
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_or_create_collection("nvda_chromadb")
chroma_vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
chroma_storage = StorageContext.from_defaults(vector_store=chroma_vector_store)

index_chroma = VectorStoreIndex(
    nodes=nodes,
    storage_context=chroma_storage,
    show_progress=True,
)
print(f"  ChromaDB index built. Vectors: {chroma_collection.count()}")

In [ ]:
# === Index B: FAISS ===
FAISS_DIR = RAG_DIR / 'vector_db_llamaindex' / 'faiss'
if FAISS_DIR.exists():
    shutil.rmtree(FAISS_DIR)
FAISS_DIR.mkdir(parents=True, exist_ok=True)

EMBED_DIM = len(test_vec)
print(f"Building FAISS index (dim={EMBED_DIM})...")

faiss_index = faiss.IndexFlatL2(EMBED_DIM)  # brute-force baseline
faiss_vector_store = FaissVectorStore(faiss_index=faiss_index)
faiss_storage = StorageContext.from_defaults(vector_store=faiss_vector_store)

index_faiss = VectorStoreIndex(
    nodes=nodes,
    storage_context=faiss_storage,
    show_progress=True,
)
print(f"  FAISS index built. Vectors: {faiss_index.ntotal}")

# Persist FAISS metadata separately
faiss_storage.persist(persist_dir=str(FAISS_DIR))

In [ ]:
# === Index C: Native LlamaIndex (JSON persistence, no external vector store) ===
NATIVE_DIR = RAG_DIR / 'vector_db_llamaindex' / 'native'
if NATIVE_DIR.exists():
    shutil.rmtree(NATIVE_DIR)
NATIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Building native LlamaIndex index (default in-memory + JSON)...")
native_storage = StorageContext.from_defaults()  # uses SimpleVectorStore in-memory
index_native = VectorStoreIndex(
    nodes=nodes,
    storage_context=native_storage,
    show_progress=True,
)
native_storage.persist(persist_dir=str(NATIVE_DIR))
print(f"  Native index built and persisted to {NATIVE_DIR}")

## 4. Loading Persisted Indexes

Each index type reloads differently. LlamaIndex abstracts the reload logic behind `load_index_from_storage`.

In [ ]:
# Reload ChromaDB index
reloaded_chroma_vectore_store = ChromaVectorStore(chroma_collection=chroma_client.get_collection("nvda_chromadb"))
reloaded_chroma_storage = StorageContext.from_defaults(
    vector_store=reloaded_chroma_vectore_store
)
reloaded_index_chroma = VectorStoreIndex.from_storage(
    storage_context=reloaded_chroma_storage
)
print(f"Reloaded ChromaDB index")

# Reload FAISS index
reloaded_faiss_vector_store = FaissVectorStore.from_persist_dir(str(FAISS_DIR))
reloaded_faiss_storage = StorageContext.from_defaults(
    vector_store=reloaded_faiss_vector_store,
    persist_dir=str(FAISS_DIR),
)
reloaded_index_faiss = load_index_from_storage(
    storage_context=reloaded_faiss_storage
)
print(f"Reloaded FAISS index")

# Reload native index
reloaded_native_storage = StorageContext.from_defaults(persist_dir=str(NATIVE_DIR))
reloaded_index_native = load_index_from_storage(
    storage_context=reloaded_native_storage
)
print(f"Reloaded native index")

## 5. Retrieval Strategies

LlamaIndex provides multiple retrieval modes out of the box. We demonstrate six approaches.

In [ ]:
question = "What does the report say about NVIDIA's data center revenue?"

# --- 5a. Basic query engine (similarity top-k) ---
print("=== 5a. Basic query engine (top_k=4) ===")
query_engine = reloaded_index_native.as_query_engine(similarity_top_k=4)
response = query_engine.query(question)
print(f"Answer: {response}")
print(f"Sources: {len(response.source_nodes)} nodes")

# --- 5b. Retriever with scores ---
print("\n=== 5b. Retriever with similarity scores ===")
retriever = reloaded_index_native.as_retriever(similarity_top_k=4)
nodes_with_score = retriever.retrieve(question)
for i, n in enumerate(nodes_with_score):
    print(f"  [{i+1}] score={n.score:.4f} | {n.text[:80]}...")

# --- 5c. MMR (diversity) ---
print("\n=== 5c. MMR query engine (diversity) ===")
mmr_engine = reloaded_index_native.as_query_engine(
    vector_store_query_mode="mmr",
    similarity_top_k=4,
    vector_store_kwargs={"mmr_lambda": 0.7},
)
mmr_response = mmr_engine.query(question)
print(f"Answer: {mmr_response}")

# --- 5d. Metadata filter ---
print("\n=== 5d. With metadata filter ===")
filters = MetadataFilters(
    filters=[
        MetadataFilter(
            key="page_label",
            operator=FilterOperator.GTE,
            value="1",
        ),
        MetadataFilter(
            key="page_label",
            operator=FilterOperator.LTE,
            value="3",
        ),
    ]
)
filtered_engine = reloaded_index_native.as_query_engine(
    similarity_top_k=4,
    filters=filters,
)
filtered_response = filtered_engine.query(question)
print(f"Answer (pages 1-3 only): {filtered_response}")

### 5e. Query Fusion (Hybrid Retrieval)

LlamaIndex's `QueryFusionRetriever` runs multiple retrievers and fuses their results — similar to hybrid retrieval in the other notebooks, but built-in.

In [ ]:
print("=== 5e. QueryFusionRetriever (hybrid) ===")

# Create two retrievers from the same index with different query modes
retriever_dense = reloaded_index_native.as_retriever(
    similarity_top_k=6,
    vector_store_query_mode="default",  # pure similarity
)
retriever_mmr = reloaded_index_native.as_retriever(
    similarity_top_k=6,
    vector_store_query_mode="mmr",
    vector_store_kwargs={"mmr_lambda": 0.5},
)

# Fuse them: reciprocal rank fusion
fusion_retriever = QueryFusionRetriever(
    retrievers=[retriever_dense, retriever_mmr],
    similarity_top_k=4,
    num_queries=1,  # use the original query (no expansion)
    mode="reciprocal_rerank",  # RRF fusion
    use_async=False,
)

fusion_engine = RetrieverQueryEngine.from_args(fusion_retriever)
fusion_response = fusion_engine.query(question)
print(f"Fusion answer: {fusion_response}")
print(f"Fusion sources: {len(fusion_response.source_nodes)} nodes")

### 5f. Reranking with Cross-Encoder

LlamaIndex supports post-retrieval rerankers. A cross-encoder scores each (query, chunk) pair independently — slower but more accurate than embedding similarity.

In [ ]:
print("=== 5f. Reranking with cross-encoder ===")

# We need a SentenceTransformer reranker model
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",  # small, fast
    top_k=3,
)

reranked_engine = reloaded_index_native.as_query_engine(
    similarity_top_k=8,  # retrieve broadly
    node_postprocessors=[reranker],  # rerank to top 3
)
reranked_response = reranked_engine.query(question)
print(f"Reranked answer: {reranked_response}")
print(f"Reranked sources: {len(reranked_response.source_nodes)} nodes")
for i, n in enumerate(reranked_response.source_nodes):
    print(f"  [{i+1}] score={n.score:.4f} | {n.text[:80]}...")

## 6. Index Optimisation

LlamaIndex optimisation focuses on three levers:
1. **Chunk size / overlap** — affects retrieval granularity.
2. **Top-k and reranking** — controls how much context reaches the LLM.
3. **Vector store choice** — FAISS for speed, ChromaDB for persistence, native for simplicity.

We benchmark the three index types below.

In [ ]:
test_queries = [
    "What is NVIDIA's revenue growth?",
    "Data center business performance",
    "Gaming segment highlights",
    "Forward guidance and outlook",
    "Stock buyback and dividends",
]

def benchmark_llamaindex(name, index, queries):
    engine = index.as_query_engine(similarity_top_k=4)
    times = []
    for q in queries:
        start = time.perf_counter()
        _ = engine.query(q)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    avg = np.mean(times) * 1000
    print(f"  {name:25s}  avg {avg:6.1f} ms")

print("Benchmarking LlamaIndex backends (5 queries each):")
benchmark_llamaindex("ChromaDB", reloaded_index_chroma, test_queries)
benchmark_llamaindex("FAISS (FlatL2)", reloaded_index_faiss, test_queries)
benchmark_llamaindex("Native (Simple)", reloaded_index_native, test_queries)

# Demonstrate speed/accuracy trade-off with top_k
print("\nEffect of top_k on query time (Native index):")
for k in [2, 4, 8, 16]:
    engine = reloaded_index_native.as_query_engine(similarity_top_k=k)
    start = time.perf_counter()
    for q in test_queries:
        _ = engine.query(q)
    elapsed = time.perf_counter() - start
    avg_ms = (elapsed / len(test_queries)) * 1000
    print(f"  top_k={k:2d}  avg {avg_ms:6.1f} ms")

## 7. Connecting an LLM with RAG

LlamaIndex's `as_query_engine` already wires the retriever, LLM, and prompt together. We customise it below to show how RAG optimises the LLM context.

**How RAG optimises context:**
1. Only the top-k nodes (chunks) reach the LLM, not the full document.
2. Reranking further reduces the context to the strongest evidence.
3. The `similarity_cutoff` postprocessor discards low-scoring chunks entirely.

In [ ]:
from llama_index.core.postprocessor import SimilarityPostprocessor

# Build an optimised query engine
optimised_engine = reloaded_index_native.as_query_engine(
    similarity_top_k=6,              # retrieve broadly
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.5),  # drop weak matches
        SentenceTransformerRerank(
            model="cross-encoder/ms-marco-MiniLM-L-2-v2",
            top_k=3,  # rerank to top 3
        ),
    ],
)

question = "Summarise NVIDIA's data center revenue performance and outlook."
response = optimised_engine.query(question)

print("=== Optimised RAG Answer ===")
print(response)
print(f"\nSources used: {len(response.source_nodes)} nodes")
for i, n in enumerate(response.source_nodes):
    tokens = len(n.text.split())
    print(f"  [{i+1}] score={n.score:.4f} | {tokens} tokens | {n.text[:60]}...")

## 8. How RAG Optimises LLM Context

LlamaIndex makes the optimisation transparent. Compare the token counts:

In [ ]:
full_text = ' '.join(n.text for n in nodes)
full_tokens = len(full_text.split())

# Simulate what the optimised engine sends
rag_nodes = retriever.retrieve(question)
rag_tokens = sum(len(n.text.split()) for n in rag_nodes[:3])  # after rerank

print(f"Full document tokens:       ~{full_tokens}")
print(f"Optimised RAG tokens:      ~{rag_tokens}")
print(f"Reduction:                 ~{100 * (1 - rag_tokens / full_tokens):.0f}%")
print()
print("With LlamaIndex, you can swap the vector store, embedding model,")
print("and LLM without changing a single line of pipeline code.")

## Summary

- **LlamaIndex** provides a unified API over 30+ vector stores, embedding models, and LLMs.
- **Index types**: FAISS (fast), ChromaDB (persistent), Native JSON (simple) — all interchangeable via `StorageContext`.
- **Retrieval strategies**: basic similarity, MMR, metadata filtering, `QueryFusionRetriever` (hybrid reciprocal rank fusion), cross-encoder reranking.
- **Optimisation**: tune chunk size, top-k, similarity cutoff, reranker model, vector store backend.
- **RAG with LLM**: the framework handles the full pipeline — retrieve, rerank, prompt, generate — with minimal boilerplate.